# Introduction

A long-slit spectrograph does something conceptually simple: it spreads a source's light out
by wavelength and records the result on a detector. What comes off the detector, though, is
not a spectrum. It is a two-dimensional image in which one axis runs roughly along wavelength,
the other runs along the slit, and neither is quite aligned with the pixel grid. The sky
contributes light everywhere. Nothing in the image says which pixel corresponds to which
wavelength, and the detector has added its own noise and offsets on top.

Turning that image into a one-dimensional spectrum, flux against wavelength, is
**spectroscopic data reduction**, and it is what this book is about.

[`specreduce`](https://github.com/astropy/specreduce) is the Astropy-coordinated package for
that job. It supplies the building blocks: finding where the spectrum falls on the detector,
estimating and removing the sky background, collapsing the 2D image to 1D, and attaching a
wavelength scale derived from arc-lamp exposures. This book works through each block in turn
on real data, then composes them into a single scripted reduction.

## Two ways to use `specreduce`

`specreduce` is built for **both interactive work and automated pipelines**, and the same
objects serve both. Interactively you fit a trace, look at it, widen a background window,
check wavelength residuals by eye, and refit. In a pipeline you chain the identical calls
with no plots and no human in the loop.

The book teaches both, and in that order. Each step is introduced interactively, with figures,
so you can see what the operation does and how to tell when it has gone wrong. The final
chapter chains the whole sequence into a non-interactive reduction. Wavelength calibration is
the one step where the two styles differ enough to warrant separate chapters
([10](10-wavelength-solution-interactive.ipynb) and
[11](11-wavelength-solution-automated.ipynb)).

## How the book is organized

The chapters follow the order of an actual reduction, so each one starts where the previous
one finished.

**Foundations**

- [2. Spectral Images](02-spectral-images.ipynb): the dispersion and cross-dispersion axes,
  and how a frame is represented in code.
- [3. Spectrum Objects](03-spectrum-objects.ipynb): the `specutils` `Spectrum` containers
  that every `specreduce` operation consumes and returns.

**From frame to spectrum**

- [4. Tracing the Spectrum](04-tracing.ipynb): locating the source across the detector.
- [5. Arc Lamps](05-arc-lamps.ipynb): the wavelength-reference frames, and the distortion
  they make visible.
- [6. Tilt Correction](06-tilt-correction.ipynb): modeling and removing the tilt of spectral
  features across the slit, so that a column is a single wavelength. *Optional: whether you
  need it depends on your instrument.*
- [7. Background Subtraction](07-background.ipynb): estimating and removing the sky.
- [8. Spectral Extraction](08-extraction.ipynb): collapsing 2D to 1D, by summed aperture or
  optimal weighting.

**Wavelength calibration**

- [9. Line Identification](09-line-identification.ipynb): detecting arc emission lines and
  identifying them by hand against reference figures.
- [10. Interactive Wavelength Solution](10-wavelength-solution-interactive.ipynb) and
  [11. Automated Wavelength Solution](11-wavelength-solution-automated.ipynb): turning the identified
  lines into a wavelength solution by hand, and fitting one automatically from prior
  knowledge of the instrument setup.
- [12. The 2D Wavelength Solution](12-wavelength-solution-2d.ipynb): chaining the tilt and
  wavelength solutions into one object covering the whole detector.

**Flux calibration**

- [13. Flux Calibration](13-flux-calibration.ipynb): turning counts into physical flux. *A
  placeholder: the tools arrive in `specreduce` v1.11.*

**End to end**

- [14. Full Pipeline](14-full-pipeline.ipynb): every step chained into one reduction.

We put tilt correction at Chapter 6, before anything is measured from the frame, because
rectification rewrites the 2D image and every later step assumes that wavelength does not vary
along the slit. That is the order a real reduction has to run in.

:::{note}
**Scope.** This book covers the steps `specreduce` itself performs. Detector-level reduction
(dark and flat correction, cosmic-ray rejection) belongs to
[`ccdproc`](https://ccdproc.readthedocs.io) and
[`astroscrappy`](https://astroscrappy.readthedocs.io) and is not taught here. The frames in
this book have already been through it: they arrive bias-subtracted, flat-fielded, and in
electrons, with a per-pixel uncertainty attached. The only detector-level step the book performs
itself is the trim that Chapter 2 explains.

Flux calibration is in scope but not yet demonstrated: `specreduce` gains the machinery for it
in **v1.11**, and [Chapter 13](13-flux-calibration.ipynb) is a placeholder until then. The book
therefore ends at a background-subtracted, extracted, wavelength-calibrated spectrum in
instrumental units.
:::

## The dataset

Every chapter works on the same data, so nothing has to be re-explained and each chapter can pick up
the previous one's output.

The frames come from [OSIRIS](https://www.gtc.iac.es/instruments/osiris/) on the 10.4 m [Gran
Telescopio Canarias](https://www.gtc.iac.es/) at the Roque de los Muchachos Observatory on La
Palma, taken in July 2014 with the R1000R grism. The science frame is a single 12-second long-slit
exposure of the TrES-3 system, from a transmission-spectroscopy time series of the transiting hot
Jupiter TrES-3b, and it comes with three arc-lamp exposures (HgAr, Ne and Xe). A master bias from
30 exposures and a master flat from 100 lamp exposures through the same grism were used to reduce
all four frames; they are not part of the dataset.

There are two reasons for choosing this dataset. The respectable one is that it shows, in one
small set, most of what long-slit spectroscopy throws at you: a tilt of the spectral lines severe
enough to see by eye, a sky background that has to be modeled and removed, a wide slit for the
science and a narrow one for the arcs, and a wavelength range that no single arc lamp covers. A
dataset that misbehaves in every instructive way is a better teacher than one that does not.

The less respectable reason is that I (H. Parviainen) hold a deep personal grudge against this
dataset. In 2014 I reduced this time series with a pipeline of my own, and its tilt correction was
not good enough. The transmission spectrum that came out showed two features far too large to be
physical: a Rayleigh-like rise in extinction toward the blue, and enhanced absorption around the
potassium resonance doublet near 767 nm. [Parviainen et al.
(2016)](https://ui.adsabs.harvard.edu/abs/2016A%26A...585A.114P/abstract) traces the potassium
feature to variability in the telluric oxygen absorption, then spends a good part of its length
wondering about the Rayleigh-like slope, which it leaves unexplained. In hindsight the culprit was
the tilt correction, and [Chapter 6](06-tilt-correction.ipynb) shows what it should have looked
like. This book is, among other things, the reduction I wish I had run.

The science exposure used a **40-arcsecond slit**, far wider than the seeing disk, and
deliberately so. Transmission spectroscopy measures transit depths that differ by a few hundred
parts per million from one passband to the next, against a comparison star observed through the
same slit (on the second CCD, which this dataset leaves out). A narrow slit would clip a seeing-
and guiding-dependent fraction of each star's light, differently for the two stars, and leave
time-variable, wavelength-dependent scatter in their ratio; a slit wide enough to lose no light
avoids it. The arcs were taken through a 1.23-arcsecond slit, so their lines are sharp enough to
calibrate against.

OSIRIS used two E2V CCD44-82 detectors (it was upgraded to OSIRIS+ in 2023). To keep the data
small, the frames here come from one CCD, binned and windowed to 512 × 1024 pixels: 512 along the
slit and 1024 along the dispersion. A helper module keeps the loading in one place:

In [ ]:
import sys

sys.path.append("../src")

import matplotlib.pyplot as plt
from astropy.visualization import simple_norm

from common import read_data

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

In [ ]:
# The frames as recorded; Chapter 2 explains the trim.
arcs, lamps, obj = read_data(trim=False)

frames = arcs + [obj]
labels = lamps + ("TrES-3 (science)",)

print(lamps)
print(obj.data.shape, obj.unit)
print(f"read noise: {obj.meta['RDNOISE']:.1f} e-")

`read_data` returns the three arc frames and the science frame as
[`CCDData`](https://docs.astropy.org/en/stable/nddata/ccddata.html) objects, each carrying a
per-pixel uncertainty and its FITS header. The detector-level reduction is done: the bias is
removed, the pixel response is flat-fielded, and the counts are converted to electrons with the
`GAIN` keyword, so photon statistics apply to the values directly. The uncertainty combines that
shot noise with the read noise, about 9 e⁻ per pixel, measured from the prescan of each frame and
stored in the `RDNOISE` keyword. `specreduce` reads it to set line-detection thresholds and
extraction weights, so a frame loaded without one quietly degrades everything downstream.

The four frames:

In [ ]:
fig, axs = plt.subplots(4, figsize=(8, 6.5), sharex="all", constrained_layout=True)

for ax, frame, label in zip(axs, frames, labels):
    ax.imshow(
        frame.data,
        origin="lower",
        aspect="auto",
        cmap=plt.cm.Blues,
        norm=simple_norm(frame.data, stretch="log", vmin=0, vmax=250_000),
    )
    ax.text(0.01, 0.92, label, va="top", ha="left", transform=ax.transAxes)

plt.setp(axs, ylabel="Cross-dispersion [pix]")
plt.setp(axs[-1], xlabel="Dispersion [pix]");

The arc frames are the wavelength reference: each lamp emits at sharp, laboratory-known
wavelengths, so its exposure fills the slit with lines whose wavelengths are tabulated. Three lamps
are used because none covers the range on its own: HgAr has the bluest bright line, Ne is dense in
the middle, and Xe covers the red. Chapters [5](05-arc-lamps.ipynb) and
[9](09-line-identification.ipynb) through [11](11-wavelength-solution-automated.ipynb) turn those
lines into a wavelength scale.

The science frame looks almost empty by comparison. The bright horizontal band near row 275 is the
target's spectrum, and the diffuse glow filling the rest of the slit is sky. Several much fainter
traces come from other objects that fell in the slit; nobody asked them, but a 40-arcsecond slit
collects whatever it is pointed at.

## What the frames still need

Three problems are visible in that figure, and they set up most of the book.

**The spectral lines are tilted.** A column of pixels ought to be a single wavelength, but the
lines lean and curve, increasingly toward the edges of the slit: an optical distortion that makes
wavelength vary along the cross-dispersion axis. It is easier to see zoomed in:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3), constrained_layout=True)

ne = arcs[lamps.index("Ne")].data
ax.imshow(
    ne,
    origin="lower",
    aspect="auto",
    cmap=plt.cm.Blues,
    interpolation="bicubic",
    norm=simple_norm(ne, stretch="log", vmin=0, vmax=150_000),
)
ax.set(
    xlim=(250, 420),
    ylim=(20, 500),
    xlabel="Dispersion [pix]",
    ylabel="Cross-dispersion [pix]",
)
# Ticks on the top spine as well, so the lean can be read off at both ends of the slit.
ax.tick_params(axis="x", top=True, labeltop=True);

Across the slit a line shifts by several pixels, far more than its own width. A wavelength solution
fitted at one row is then wrong at another, and sky subtraction suffers worse: the sky rows
subtracted from the target's row sample slightly different wavelengths, so wherever the sky
spectrum has sharp structure the subtraction leaves a residual, and that residual changes from
exposure to exposure if the target drifts along the slit during the night.
[Chapter 6](06-tilt-correction.ipynb) fits a 2D model of the distortion and *rectifies* the frames
so that columns become wavelengths again.

**There is no wavelength scale.** Nothing in the frames says which pixel is which wavelength. A
crude cut through the science trace, the 11 rows around row 275 summed, makes the point:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.6), constrained_layout=True)

ax.plot(obj.data[270:281].sum(axis=0), lw=0.8)
ax.set(xlabel="Dispersion [pix]", ylabel="Signal [e$^-$]", xlim=(0, obj.shape[1]));

This is recognizably a spectrum: the broad hump is the combined response of the star, the grism and
the detector, and the dips are real absorption, some stellar and some telluric. But the horizontal
axis is pixels, so the features cannot yet be identified, and the vertical axis still includes the
sky.

**The sky is still in there.** The cut sums the target rows and the sky on top of them.
[Chapter 7](07-background.ipynb) estimates the sky from rows either side of the trace and subtracts
it, and [Chapter 8](08-extraction.ipynb) does the collapse to 1D properly, weighting pixels by their
signal-to-noise instead of summing a fixed block of rows. By [Chapter 14](14-full-pipeline.ipynb)
this same cut has become a background-subtracted, optimally extracted spectrum on a calibrated
wavelength axis, produced by a script rather than by hand.

## Running these notebooks

Every chapter is an executable notebook. To run them you need `specreduce` and its dependencies
(see `requirements.txt` in the repository) and the data files under `data/`, which
`common.read_data` locates on its own. The chapters assume you have read the ones before them and
reuse the same variable names (`arcs`, `lamps`, `obj`), but each reloads the data from scratch, so
any one of them runs top to bottom by itself.

Next: [Spectral Images](02-spectral-images.ipynb), the 2D frame and its two axes.